In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import json

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import json


Phase 1 – Dataset Loading

In [ ]:
#1
base_path = "/kaggle/input/2017-2017"

print("Base dataset folder:")
print(base_path)

print("\nSubfolders:")
for item in os.listdir(base_path):
    item_path = os.path.join(base_path, item)
    if os.path.isdir(item_path):
        print("  └──", item)


In [ ]:
#2 
import os

BASE_PATH = "/kaggle/input/2017-2017"

TRAIN_IMAGES_PATH = os.path.join(BASE_PATH, "train2017")
VAL_IMAGES_PATH   = os.path.join(BASE_PATH, "val2017")

ANNOTATIONS_PATH  = os.path.join(BASE_PATH, "annotations_trainval2017", "annotations")

TRAIN_CAPTIONS_FILE = os.path.join(ANNOTATIONS_PATH, "captions_train2017.json")
VAL_CAPTIONS_FILE   = os.path.join(ANNOTATIONS_PATH, "captions_val2017.json")

print("TRAIN_IMAGES_PATH exists:", os.path.exists(TRAIN_IMAGES_PATH))
print("VAL_IMAGES_PATH exists  :", os.path.exists(VAL_IMAGES_PATH))
print("Train captions exists   :", os.path.exists(TRAIN_CAPTIONS_FILE))
print("Val captions exists     :", os.path.exists(VAL_CAPTIONS_FILE))

required_paths = [
    TRAIN_IMAGES_PATH,
    VAL_IMAGES_PATH,
    TRAIN_CAPTIONS_FILE,
    VAL_CAPTIONS_FILE,
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Missing required dataset paths:\n" + "\n".join(missing))


PHASE 2: DATA UNDERSTANDING

STEP 1: Load the Training Captions JSON

In [ ]:
#3
import json
import os

if not os.path.exists(TRAIN_CAPTIONS_FILE):
    raise FileNotFoundError(f"Missing file: {TRAIN_CAPTIONS_FILE}")

with open(TRAIN_CAPTIONS_FILE, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Top-level keys in training annotation file:")
print(train_data.keys())

STEP 3: Inspect Images Metadata

In [ ]:
#4
print("\nSample image entry:")
if "images" not in train_data or len(train_data["images"]) == 0:
    raise ValueError("train_data['images'] is missing or empty.")
print(train_data["images"][0])

In [ ]:
# 5
if not os.path.exists(VAL_CAPTIONS_FILE):
    raise FileNotFoundError(f"Missing file: {VAL_CAPTIONS_FILE}")

with open(VAL_CAPTIONS_FILE, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print("Number of training images:", len(train_data.get("images", [])))
print("Number of validation images:", len(val_data.get("images", [])))

STEP 4.1: Count Total Captions

In [ ]:
#6
print("Number of captions:", len(train_data.get("annotations", [])))
print("Sample caption entry:")

if "annotations" not in train_data or len(train_data["annotations"]) == 0:
    raise ValueError("train_data['annotations'] is missing or empty.")

print(train_data["annotations"][0])

STEP 4.3: Verify Image → Multiple Captions

In [ ]:
#7
from collections import defaultdict

if "annotations" not in train_data or not isinstance(train_data["annotations"], list):
    raise ValueError("train_data['annotations'] is missing or invalid.")

image_caption_count = defaultdict(int)

for ann in train_data["annotations"]:
    image_id = ann.get("image_id")
    if image_id is not None:
        image_caption_count[image_id] += 1

# Check a few images
print(list(image_caption_count.items())[:5])



STEP 5: IMAGE → CAPTIONS MAPPING

STEP 5.1: Map image_id → file_name

In [ ]:
#8
# Create image_id → filename mapping
if "images" not in train_data or not isinstance(train_data["images"], list):
    raise ValueError("train_data['images'] is missing or invalid.")

image_id_to_filename = {}

for img in train_data["images"]:
    img_id = img.get("id")
    file_name = img.get("file_name")
    if img_id is not None and file_name is not None:
        image_id_to_filename[img_id] = file_name

print("Total image IDs mapped:", len(image_id_to_filename))


STEP 5.2: Build filename → captions Dictionary

In [ ]:
#9
from collections import defaultdict

if "annotations" not in train_data or not isinstance(train_data["annotations"], list):
    raise ValueError("train_data['annotations'] is missing or invalid.")

image_to_captions = defaultdict(list)

for ann in train_data["annotations"]:
    image_id = ann.get("image_id")
    caption = ann.get("caption")
    filename = image_id_to_filename.get(image_id)

    if filename is not None and caption is not None:
        image_to_captions[filename].append(caption)

print("Total images with captions:", len(image_to_captions))



STEP 5.3: Inspect One Image’s Captions

In [ ]:
#10
if len(image_to_captions) == 0:
    raise ValueError("image_to_captions is empty. Check previous mapping cell.")

sample_image = next(iter(image_to_captions.keys()))

print("Image filename:", sample_image)
print("Captions:")
for cap in image_to_captions[sample_image]:
    print("-", cap)



EDA STEP 1: SAMPLE IMAGES WITH CAPTIONS (MANDATORY)

In [ ]:
#11
import matplotlib.pyplot as plt

# Derive counts from loaded COCO annotation dicts
train_img_count = len(train_data.get("images", []))
val_img_count = len(val_data.get("images", []))

train_cap_count = len(train_data.get("annotations", []))
val_cap_count = len(val_data.get("annotations", []))

splits = ['Train', 'Validation']
num_images = [train_img_count, val_img_count]
num_captions = [train_cap_count, val_cap_count]

bar_width = 0.35
x = range(len(splits))

plt.figure(figsize=(8, 5))
plt.bar([i - bar_width/2 for i in x], num_images, width=bar_width, label='Images', color='skyblue')
plt.bar([i + bar_width/2 for i in x], num_captions, width=bar_width, label='Captions', color='orange')

plt.xticks(list(x), splits)
plt.ylabel('Count')
plt.title('Number of Images vs Captions per Split')
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
#12

import os
import matplotlib.pyplot as plt
from PIL import Image
import random
import textwrap

# Dependencies expected from earlier cells:
# - train_data
# - TRAIN_IMAGES_PATH
# - image_to_captions (filename -> list of captions), created in Cell #9

if "images" not in train_data or len(train_data["images"]) == 0:
    raise ValueError("train_data['images'] is missing or empty.")

sample_n = min(2, len(train_data["images"]))
sample_images = random.sample(train_data["images"], sample_n)

plt.figure(figsize=(14, 6))

for i, img_info in enumerate(sample_images):
    file_name = img_info.get("file_name")
    if file_name is None:
        continue

    img_path = os.path.join(TRAIN_IMAGES_PATH, file_name)
    if not os.path.exists(img_path):
        continue

    # Prefer prebuilt mapping from Cell #9 (fast)
    captions = image_to_captions.get(file_name, [])

    image = Image.open(img_path).convert("RGB")

    plt.subplot(1, sample_n, i + 1)
    plt.imshow(image)
    plt.axis("off")

    if captions:
        caption_text = "\n".join(
            [f"{j+1}. {textwrap.fill(cap, 45)}" for j, cap in enumerate(captions)]
        )
    else:
        caption_text = "No captions found."

    plt.title(caption_text, fontsize=9, loc="left")

plt.tight_layout()
plt.show()



PHASE 3: PREPROCESSING

STEP 3.1.1: Text Cleaning Function
What this does

Lowercase

Remove punctuation

Remove extra spaces

In [ ]:
#13
import re

def clean_caption(caption):
    if not isinstance(caption, str):
        return ""
    caption = caption.lower()
    caption = re.sub(r"[^a-z\s]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption



STEP 3.1.2: Add <start> and <end> Tokens

In [ ]:
#14
def add_tokens(caption):
    return f"<start> {caption} <end>"




STEP 3.1.3: Apply Cleaning to All Training Captions

In [ ]:
#15
all_captions = []

for captions in image_to_captions.values():
    for cap in captions:
        cap = clean_caption(cap)
        cap = add_tokens(cap)
        all_captions.append(cap)

print("Total processed captions:", len(all_captions))
if len(all_captions) == 0:
    raise ValueError("No captions were processed. Check image_to_captions.")
print("Sample caption:", all_captions[0])


PHASE 3.2: TOKENIZATION & VOCABULARY

STEP 3.2.1: Tokenize Captions

In [ ]:
#16
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<unk>",
    filters=""
)

tokenizer.fit_on_texts(all_captions)


STEP 3.2.2: Vocabulary Size

In [ ]:
#17
# Limit vocabulary to the top 10,000 most frequent words
vocab_size = min(10000, len(tokenizer.word_index) + 1)
print("Vocabulary size:", vocab_size)


STEP 3.2.3: Convert Captions to Sequences

What it does (very brief):

texts_to_sequences → turns words into numbers

max_length → finds the longest caption

pad_sequences → adds zeros so all captions have the same length

Why needed:
Neural networks require fixed-length inputs.

In [ ]:
#18
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os
import pickle

sequences = tokenizer.texts_to_sequences(all_captions)
if not sequences:
    raise ValueError("No token sequences generated. Check all_captions/tokenizer.")

max_length = max(len(seq) for seq in sequences)
print("Maximum caption length:", max_length)

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post"
)

# Resume-friendly artifact location
ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

with open(os.path.join(ARTIFACTS_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "vocab_size": vocab_size,
    "max_length": max_length
}
with open(os.path.join(ARTIFACTS_DIR, "config.pkl"), "wb") as f:
    pickle.dump(config, f)

print("Tokenizer and config saved successfully.")


In [ ]:
#savepoint 
import pickle

# Save tokenizer
pickle.dump(tokenizer, open("/kaggle/working/tokenizer.pkl", "wb"))

# Save important config
config = {
    "vocab_size": vocab_size,
    "max_length": max_length
}

pickle.dump(config, open("/kaggle/working/config.pkl", "wb"))

print("Tokenizer and config saved successfully.")


PHASE 3.3: IMAGE PREPROCESSING (ResNet-101)

In [ ]:
#19
from tensorflow.keras.applications.resnet import ResNet101, preprocess_input
from tensorflow.keras.models import Model

# Explicit input shape for COCO preprocessing pipeline
base_model = ResNet101(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

encoder = Model(
    inputs=base_model.input,
    outputs=base_model.output  # shape: (None, 7, 7, 2048)
)

encoder.trainable = False


STEP 3.3.2: Image Preprocessing Function

In [ ]:
#20
import numpy as np
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet import preprocess_input

IMAGE_SIZE = (224, 224)

def preprocess_image(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    img = load_img(image_path, target_size=IMAGE_SIZE)
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    return img



In [ ]:
#21
import matplotlib.pyplot as plt
from collections import Counter
import random

if not image_to_captions:
    raise ValueError("image_to_captions is empty. Run mapping cell first.")

sample_filename = random.choice(list(image_to_captions.keys()))
sample_captions = image_to_captions[sample_filename]

if not sample_captions:
    raise ValueError(f"No captions found for {sample_filename}")

# Caption length plot
caption_lengths = [len(cap.split()) for cap in sample_captions]
plt.figure(figsize=(6, 4))
plt.bar(range(len(sample_captions)), caption_lengths)
plt.title(f"Caption lengths for {sample_filename}")
plt.xlabel("Caption index")
plt.ylabel("Token count")
plt.tight_layout()
plt.show()

# Word frequency plot
words = [word.lower().strip(".,") for cap in sample_captions for word in cap.split()]
word_counts = Counter(words)
top_words = word_counts.most_common(10)

if top_words:
    words_list, counts_list = zip(*top_words)
    plt.figure(figsize=(8, 4))
    plt.bar(words_list, counts_list)
    plt.title("Top-10 word frequencies (sample image captions)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
#22
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Prefer previously defined path from setup cells
image_folder = TRAIN_IMAGES_PATH  # e.g., /kaggle/input/2017-2017/train2017

if not os.path.isdir(image_folder):
    raise FileNotFoundError(f"Image folder not found: {image_folder}")

# COCO train2017 is flat; deterministic file selection
jpg_files = sorted(
    f for f in os.listdir(image_folder)
    if f.lower().endswith(".jpg")
)

if not jpg_files:
    raise ValueError("No .jpg images found in the dataset!")

image_path = os.path.join(image_folder, jpg_files[0])
print("Using image:", image_path)

# Open the original image and convert to RGB
img_original = Image.open(image_path).convert("RGB")

# Resize to 224x224
img_resized = img_original.resize((224, 224))

# Convert to numpy arrays for plotting
orig_array = np.array(img_original)
resized_array = np.array(img_resized)

# Plot side by side
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(orig_array)
plt.title(f"Original: {img_original.size[0]}x{img_original.size[1]}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(resized_array)
plt.title("Resized: 224x224")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#23
import os
import pickle
import numpy as np
from tqdm import tqdm

# Reuse earlier variable from setup cell
TRAIN_IMG_DIR = TRAIN_IMAGES_PATH
ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
FEATURES_PATH = os.path.join(ARTIFACTS_DIR, "image_features_5k.pkl")

if not os.path.isdir(TRAIN_IMG_DIR):
    raise FileNotFoundError(f"Training image directory not found: {TRAIN_IMG_DIR}")

# Use deterministic subset for reproducibility
image_list = sorted(image_to_captions.keys())[:5000]

# Resume-friendly: load existing feature cache if present
if os.path.exists(FEATURES_PATH):
    with open(FEATURES_PATH, "rb") as f:
        image_features = pickle.load(f)
else:
    image_features = {}

batch_size = 32
pending = [fn for fn in image_list if fn not in image_features]

for start in tqdm(range(0, len(pending), batch_size), desc="Extracting features"):
    batch_filenames = pending[start:start + batch_size]
    batch_imgs = []
    valid_filenames = []

    for filename in batch_filenames:
        img_path = os.path.join(TRAIN_IMG_DIR, filename)
        if not os.path.exists(img_path):
            continue

        img = preprocess_image(img_path)  # (1,224,224,3)
        batch_imgs.append(img[0])
        valid_filenames.append(filename)

    if not batch_imgs:
        continue

    batch_array = np.stack(batch_imgs, axis=0)  # (B,224,224,3)
    batch_features = encoder.predict(batch_array, verbose=0)  # (B,7,7,2048)

    if batch_features.ndim != 4 or batch_features.shape[1:] != (7, 7, 2048):
        raise ValueError(f"Unexpected encoder output shape: {batch_features.shape}")

    batch_features = batch_features.reshape(batch_features.shape[0], 49, 2048)

    for i, filename in enumerate(valid_filenames):
        image_features[filename] = batch_features[i]

    # periodic save for crash-safe resume
    with open(FEATURES_PATH, "wb") as f:
        pickle.dump(image_features, f)

print("Features saved. Total:", len(image_features))
print("Saved to:", FEATURES_PATH)


In [ ]:
#24
import os
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

ARTIFACTS_DIR = "/kaggle/working/artifacts"
FEATURES_PATH = os.path.join(ARTIFACTS_DIR, "image_features_5k.pkl")
TRAIN_DATA_PATH = os.path.join(ARTIFACTS_DIR, "train_arrays_5k.pkl")

if not os.path.exists(FEATURES_PATH):
    raise FileNotFoundError(f"Missing features file: {FEATURES_PATH}")

with open(FEATURES_PATH, "rb") as f:
    image_features = pickle.load(f)

X_img, X_seq, y_seq = [], [], []

for filename, caps in image_to_captions.items():
    if filename not in image_features:
        continue

    for cap in caps:
        seq = tokenizer.texts_to_sequences([add_tokens(clean_caption(cap))])[0]
        if len(seq) < 2:
            continue

        in_seq = seq[:-1]
        out_seq = seq[1:]

        in_seq = pad_sequences([in_seq], maxlen=max_length, padding="post")[0]
        out_seq = pad_sequences([out_seq], maxlen=max_length, padding="post")[0]

        X_img.append(image_features[filename])   # (49, 2048)
        X_seq.append(in_seq)                     # (max_length,)
        y_seq.append(out_seq)                    # (max_length,)

X_img = np.array(X_img, dtype=np.float32)
X_seq = np.array(X_seq, dtype=np.int32)
y_seq = np.array(y_seq, dtype=np.int32)

print("X_img shape:", X_img.shape)
print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)

with open(TRAIN_DATA_PATH, "wb") as f:
    pickle.dump({"X_img": X_img, "X_seq": X_seq, "y_seq": y_seq}, f)

print("Saved training arrays to:", TRAIN_DATA_PATH)


In [ ]:
#25
import os
import pickle
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

ARTIFACTS_DIR = "/kaggle/working/artifacts"
TRAIN_DATA_PATH = os.path.join(ARTIFACTS_DIR, "train_arrays_5k.pkl")

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f"Missing training arrays file: {TRAIN_DATA_PATH}")

with open(TRAIN_DATA_PATH, "rb") as f:
    data = pickle.load(f)

X_img = data["X_img"]   # (N, 49, 2048)
X_seq = data["X_seq"]   # (N, max_length)
y_seq = data["y_seq"]   # (N, max_length)

if len(X_img) == 0:
    raise ValueError("Training arrays are empty. Check previous cell outputs.")

X_img_train, X_img_val, X_seq_train, X_seq_val, y_train, y_val = train_test_split(
    X_img, X_seq, y_seq, test_size=0.1, random_state=42
)

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices(((X_img_train, X_seq_train), y_train))
val_ds   = tf.data.Dataset.from_tensor_slices(((X_img_val, X_seq_val), y_val))

train_ds = train_ds.shuffle(min(10000, len(X_img_train)), seed=42, reshuffle_each_iteration=True)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Train samples:", len(X_img_train))
print("Val samples:", len(X_img_val))
print("BATCH_SIZE:", BATCH_SIZE)


In [ ]:
#26
import os
import glob
import tensorflow as tf
from tensorflow.keras import layers, Model

# Expected dependencies from prior cells:
# - vocab_size
# - max_length
# - train_ds, val_ds

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EMBED_DIM = 256
UNITS = 512
FEATURE_DIM = 2048
NUM_FEATURES = 49

class BahdanauAttention(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, features, hidden_state):
        hidden_with_time_axis = tf.expand_dims(hidden_state, 1)
        score = self.V(tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = tf.reduce_sum(attention_weights * features, axis=1)
        return context_vector, attention_weights

class CaptionDecoder(layers.Layer):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.attention = BahdanauAttention(units)
        self.init_h = layers.Dense(units, activation="tanh")
        self.lstm = layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc1 = layers.Dense(units, activation="relu")
        self.dropout = layers.Dropout(0.3)
        self.fc2 = layers.Dense(vocab_size)

    def call(self, features, seq_inputs, training=False):
        x = self.embedding(seq_inputs)
        hidden0 = self.init_h(tf.reduce_mean(features, axis=1))
        cell0 = tf.zeros_like(hidden0)

        context_vector, _ = self.attention(features, hidden0)
        context_seq = tf.repeat(tf.expand_dims(context_vector, 1), repeats=tf.shape(x)[1], axis=1)
        x = tf.concat([context_seq, x], axis=-1)

        x, _, _ = self.lstm(x, initial_state=[hidden0, cell0], training=training)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        return self.fc2(x)

img_inputs = layers.Input(shape=(NUM_FEATURES, FEATURE_DIM), name="img_features")
seq_inputs = layers.Input(shape=(max_length,), name="seq_inputs")

decoder = CaptionDecoder(vocab_size=vocab_size, embed_dim=EMBED_DIM, units=UNITS)
logits = decoder(img_inputs, seq_inputs)
model = Model(inputs=[img_inputs, seq_inputs], outputs=logits)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
def masked_loss(y_true, y_pred):
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, 0), dtype=loss.dtype)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=masked_loss,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="token_acc")]
)

ckpt_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "ckpt_epoch_{epoch:02d}.weights.h5"),
    save_weights_only=True,
    save_best_only=False,
    verbose=1
)

checkpoint_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "ckpt_epoch_*.weights.h5")))
latest_ckpt = checkpoint_files[-1] if checkpoint_files else None
if latest_ckpt:
    print(f"Loading checkpoint: {latest_ckpt}")
    model.load_weights(latest_ckpt)
else:
    print("No checkpoint found. Training from scratch.")

model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[ckpt_callback]
)


In [ ]:
#27
import os
import pickle
import numpy as np
import tensorflow as tf

# Expected dependencies from prior cells:
# - model (trained caption model from Cell 26)
# - tokenizer
# - max_length
# - TRAIN_IMAGES_PATH
# - preprocess_image
# - encoder

ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Build reverse vocab for decoding token ids -> words
index_word = tokenizer.index_word
word_index = tokenizer.word_index
start_id = word_index.get("<start>")
end_id = word_index.get("<end>")
unk_token = "<unk>"

if start_id is None or end_id is None:
    raise ValueError("Tokenizer must contain <start> and <end> tokens.")


def extract_image_feature(filename):
    """Load one train image, preprocess, and encode to (49, 2048)."""
    img_path = os.path.join(TRAIN_IMAGES_PATH, filename)
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"Image not found: {img_path}")

    img = preprocess_image(img_path)                        # (1, 224, 224, 3)
    feat = encoder.predict(img, verbose=0)                 # (1, 7, 7, 2048)
    feat = feat.reshape(1, 49, 2048).astype(np.float32)    # (1, 49, 2048)
    return feat


def greedy_decode_caption(image_feature):
    """Greedy decode one caption from feature tensor of shape (1, 49, 2048)."""
    seq = [start_id]

    for _ in range(max_length - 1):
        padded = tf.keras.preprocessing.sequence.pad_sequences(
            [seq], maxlen=max_length, padding="post"
        )

        logits = model.predict([image_feature, padded], verbose=0)  # (1, max_length, vocab)
        next_id = int(np.argmax(logits[0, len(seq) - 1]))

        if next_id == 0 or next_id == end_id:
            break

        seq.append(next_id)

    words = [index_word.get(tok, unk_token) for tok in seq if tok not in (start_id, end_id, 0)]
    return " ".join(words).strip()


# Demo on one deterministic image from training folder
sample_filename = sorted([f for f in os.listdir(TRAIN_IMAGES_PATH) if f.lower().endswith(".jpg")])[0]
feature = extract_image_feature(sample_filename)
base_caption = greedy_decode_caption(feature)

print("Image:", sample_filename)
print("Generated base caption:", base_caption)

# Save sample prediction for quick resume/inspection
sample_out = {
    "image": sample_filename,
    "caption": base_caption
}

with open(os.path.join(ARTIFACTS_DIR, "sample_base_caption.pkl"), "wb") as f:
    pickle.dump(sample_out, f)

print("Saved:", os.path.join(ARTIFACTS_DIR, "sample_base_caption.pkl"))


In [ ]:
#28
import os
import pickle
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Expected dependencies from prior cells:
# - base_caption (from Cell 27) OR /kaggle/working/artifacts/sample_base_caption.pkl

ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Load base caption from memory or artifact fallback
if "base_caption" in globals() and isinstance(base_caption, str) and base_caption.strip():
    source_caption = base_caption.strip()
else:
    sample_path = os.path.join(ARTIFACTS_DIR, "sample_base_caption.pkl")
    if not os.path.exists(sample_path):
        raise FileNotFoundError(
            "No base caption found. Run Cell 27 first or ensure sample_base_caption.pkl exists."
        )
    with open(sample_path, "rb") as f:
        sample_out = pickle.load(f)
    source_caption = str(sample_out.get("caption", "")).strip()

if not source_caption:
    raise ValueError("Base caption is empty; cannot refine.")

MODEL_NAME = "google/flan-t5-small"
flan_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


def refine_for_platform(caption: str, platform: str) -> str:
    platform = platform.lower().strip()

    style_guide = {
        "instagram": "Make it catchy, emoji-friendly, and include 3 relevant hashtags.",
        "facebook": "Make it friendly, conversational, and community-oriented.",
        "linkedin": "Make it professional, concise, and insight-driven."
    }

    if platform not in style_guide:
        raise ValueError("platform must be one of: instagram, facebook, linkedin")

    prompt = (
        f"Rewrite this image caption for {platform}. "
        f"{style_guide[platform]}\n"
        f"Original caption: {caption}\n"
        f"Refined caption:"
    )

    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    outputs = flan_model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        do_sample=False,
        early_stopping=True
    )

    refined = flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return refined

platform_outputs = {
    "base": source_caption,
    "instagram": refine_for_platform(source_caption, "instagram"),
    "facebook": refine_for_platform(source_caption, "facebook"),
    "linkedin": refine_for_platform(source_caption, "linkedin"),
}

print("Base caption:", platform_outputs["base"])
print("\nInstagram:", platform_outputs["instagram"])
print("\nFacebook:", platform_outputs["facebook"])
print("\nLinkedIn:", platform_outputs["linkedin"])

out_path = os.path.join(ARTIFACTS_DIR, "refined_captions.pkl")
with open(out_path, "wb") as f:
    pickle.dump(platform_outputs, f)

print("\nSaved:", out_path)


In [ ]:
#29
import os
import pickle
import pandas as pd

# Expected dependencies from prior cells:
# - refined captions artifact from Cell 28: /kaggle/working/artifacts/refined_captions.pkl

ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

REFINED_PATH = os.path.join(ARTIFACTS_DIR, "refined_captions.pkl")
CSV_PATH = os.path.join(ARTIFACTS_DIR, "caption_outputs.csv")

if not os.path.exists(REFINED_PATH):
    raise FileNotFoundError(
        f"Missing refined captions artifact: {REFINED_PATH}. Run Cell 28 first."
    )

with open(REFINED_PATH, "rb") as f:
    refined = pickle.load(f)

required_keys = ["base", "instagram", "facebook", "linkedin"]
missing_keys = [k for k in required_keys if k not in refined]
if missing_keys:
    raise ValueError(f"Refined caption artifact missing keys: {missing_keys}")

# Create final table for easy viewing/export
result_df = pd.DataFrame(
    {
        "platform": ["base", "instagram", "facebook", "linkedin"],
        "caption": [
            str(refined["base"]).strip(),
            str(refined["instagram"]).strip(),
            str(refined["facebook"]).strip(),
            str(refined["linkedin"]).strip(),
        ],
    }
)

# Basic quality checks
if (result_df["caption"].str.len() == 0).any():
    raise ValueError("One or more generated captions are empty.")

result_df.to_csv(CSV_PATH, index=False)

print("Final caption table:")
print(result_df)
print("\nSaved CSV:", CSV_PATH)
